# 동아대 대학원 QML 강의 — 10월 준비 (3일차용)
## QGAN·CNN 체크포인트 생성 (Colab 전용)

**목적**: 3일차(11/27) 실습에서 학생들이 그대로 불러와 쓸 체크포인트 3종
(`qgan_df_8px.pt`, `cnn_before.pt`, `cnn_after.pt`, `comparison_result.json`)을
**강사가 미리 실제 HAM10000 데이터로 학습해서** Google Drive에 저장해두는 준비용 노트북입니다.
**2일차 준비(EDA·파이프라인 리허설)는 별도 노트북**(`day2/prep_day2_eda_rehearsal.ipynb`)에서
진행합니다 — 2일차는 학습이 없는 시간이라 체크포인트가 필요 없고, 이 노트북(3일차용)이 실제
무거운 학습·산출물 생성을 전담합니다.

**왜 로컬이 아니라 Colab인가요?** 이 강의 준비 작업용 로컬 PC에는 Kaggle API가 설정되어 있지 않아
HAM10000(10,015장, 약 6GB) 원본 데이터를 받을 수 없습니다. Colab에서는 `kagglehub`로
바로 받을 수 있어 이 준비 작업 전체를 여기서 진행합니다.

**2일차·3일차 실습 노트북과의 관계**: 이 노트북은 강의 전에 강사가 한 번 실행하는 준비용입니다(수업 중에는 쓰지 않음).
2일차 노트북의 세그멘테이션·인코딩 코드, 3일차 노트북의 `PatchQuantumGenerator`/`PaperCNNClassifier`
코드와 **완전히 동일한 로직**을 그대로 재사용합니다(별도로 만든 다른 코드가 아님 — 학생이 3일차에
불러오는 체크포인트의 `state_dict`가 그 클래스 정의와 반드시 일치해야 로드되기 때문입니다).

**예상 소요 시간**: QGAN 학습(epoch·데이터 규모에 따라 유동, 300epoch 기준 수 분~수십 분) +
CNN 학습(20epoch, 수 분) — 양자회로 시뮬레이션(`default.qubit`)과 8×8 CNN 모두 CPU로 실행되므로
기본 CPU 런타임으로 충분함(GPU 런타임으로 바꿔도 빨라지지 않음).

**정직한 비교 원칙**: 증강 전/후 CNN은 반드시 **같은 held-out 검증셋**으로 평가하고,
검증셋에는 합성(QGAN 생성) 이미지를 절대 섞지 않습니다 — 학습셋에만 증강을 적용합니다.


## 1. 환경 설정

In [ ]:
!pip install -q pennylane pennylane-lightning scikit-fuzzy kagglehub
print("설치 완료")

In [ ]:
import os, math, json, time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pennylane as qml
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, accuracy_score
import skfuzzy as fuzz

print("torch:", torch.__version__, "| pennylane:", qml.__version__, "| opencv:", cv2.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

## 2. HAM10000 다운로드 (kagglehub)

`kagglehub`로 다운로드(약 6GB, Colab 런타임 디스크에 저장. Colab 전용 캐시에서 바로 연결되면 수 초). HAM10000은
공개 데이터셋이라 로그인이 필요 없음.

In [ ]:
import kagglehub

In [ ]:
DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("데이터셋 경로:", DATA_DIR)

METADATA_CSV = f"{DATA_DIR}/HAM10000_metadata.csv"
IMG_DIR = DATA_DIR
metadata = pd.read_csv(METADATA_CSV)
print(f"전체 이미지 수: {len(metadata)}")
print(metadata["dx"].value_counts())

## 3. 데이터 파이프라인 (2일차 노트북과 동일 로직)

세그멘테이션·저해상도 인코딩 함수 — `day3/data_pipeline.py`와 완전히 동일합니다.

In [ ]:
def find_image_path(img_dir, image_id, exts=(".jpg", ".jpeg", ".png")):
    for ext in exts:
        for sub in ["", "HAM10000_images_part_1", "HAM10000_images_part_2"]:
            p = f"{img_dir}/{sub}/{image_id}{ext}" if sub else f"{img_dir}/{image_id}{ext}"
            if os.path.exists(p):
                return p
    return None


def segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=None):
    '''HAM10000 QGAN 논문 Section 3.1 — K-means+Fuzzy C-means 하이브리드 세그멘테이션.'''
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if resize_to is not None:
        gray = cv2.resize(gray, resize_to, interpolation=cv2.INTER_AREA)
    h, w = gray.shape
    pixels = gray.reshape(-1, 1).astype(np.float32)

    # 1단계: K-means로 대략적 클러스터링 (논문: "pre-clustering으로 FCM 처리시간 단축")
    km = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(pixels)
    # K-means 중심까지의 거리로 FCM 초기 소속도 행렬 구성 (m=2 → 소속도 ∝ 1/거리²)
    dist = np.abs(pixels - km.cluster_centers_.reshape(1, -1)) + 1e-6
    u_init = 1.0 / dist ** 2
    u_init = (u_init / u_init.sum(axis=1, keepdims=True)).T  # (클러스터 수, 픽셀 수)

    # 2단계: Fuzzy C-means로 정제 (K-means 결과에서 출발 → 무작위 시작보다 빨리 수렴, 매번 같은 결과)
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        pixels.T, c=n_clusters, m=2.0, error=1e-4, maxiter=100, init=u_init)
    membership = np.argmax(u, axis=0)
    lesion_cluster = np.argmin(cntr.flatten())  # 가장 어두운 클러스터 = 병변
    mask = (membership == lesion_cluster).reshape(h, w).astype(np.uint8) * 255
    return cv2.bitwise_and(gray, gray, mask=mask)


def resize_for_encoding(img_gray, size):
    return cv2.resize(img_gray, size, interpolation=cv2.INTER_AREA)


def build_dataset(metadata, img_dir, target_class, img_size, max_n=None):
    rows = metadata[metadata["dx"] == target_class]
    if max_n:
        rows = rows.head(max_n)
    imgs = []
    for _, row in rows.iterrows():
        path = find_image_path(img_dir, row["image_id"])
        if path is None:
            continue
        bgr = cv2.imread(path)
        seg = segment_fcm_kmeans(bgr, n_clusters=4, resize_to=(64, 64))
        imgs.append(resize_for_encoding(seg, (img_size, img_size)).astype(np.float32) / 255.0)
    return np.stack(imgs) if imgs else np.zeros((0, img_size, img_size), dtype=np.float32)

print("데이터 파이프라인 함수 정의 완료")

## 4. QGAN 모델 정의 (`qgan_model.py`와 동일 로직)

측정 방식: 보조큐빗까지 측정해 **보조큐빗=0인 항목만** patch로 사용함(논문의 ancilla 사후선택에 해당,
PennyLane 공식 QGAN 튜토리얼과 같은 방향). 단, 튜토리얼처럼 patch별로 다시 정규화하지는 않음 — patch 밝기
총합(0~1)을 생성자가 직접 학습하게 하려는 것임.

**2026-09-22 변경 이유**: 이전처럼 데이터 큐빗만 측정(보조큐빗 partial trace)하면 patch마다 밝기 총합이
항상 1로 고정됨. 세그멘테이션 이미지는 가운데 병변 외에는 검정이라 위·아래 patch 총합이 0에 가까우므로,
판별자가 이 차이만으로 항상 이겨 생성 이미지가 노이즈가 됨. 실데이터 3개 seed 비교에서 생성-실제 평균 이미지
상관계수가 0.00 → 0.49로 개선됨(README 참고).

In [ ]:
class PatchQuantumGenerator(nn.Module):
    '''서브제너레이터 앙상블 기반 양자 생성자. 원 논문 스펙: n_generators=16, n_data_qubits=7,
    n_ancillas=1, q_depth=10 (강의는 축소된 값 사용, 아래 설정 셀 참고).'''

    def __init__(self, n_generators=4, n_data_qubits=4, n_ancillas=1, q_depth=3, dev_name="default.qubit"):
        super().__init__()
        self.n_generators = n_generators
        self.n_data_qubits = n_data_qubits
        self.n_ancillas = n_ancillas
        self.n_qubits = n_data_qubits + n_ancillas
        self.q_depth = q_depth
        self.patch_dim = 2 ** n_data_qubits
        self.anc_stride = 2 ** n_ancillas  # 보조큐빗이 마지막 wire → '보조큐빗=0' 항목은 이 간격마다 위치함

        dev = qml.device(dev_name, wires=self.n_qubits)
        n_qubits = self.n_qubits

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(noise, weights):
            for i in range(n_qubits):
                qml.RY(noise[i], wires=i)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])
            # 보조큐빗까지 전체 측정 → forward에서 '보조큐빗=0' 항목만 patch로 사용 (논문의 사후선택에 해당)
            return qml.probs(wires=range(n_qubits))

        self.circuit = circuit
        self.q_params = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, self.n_qubits) * math.pi) for _ in range(n_generators)
        ])

    def forward(self, batch_size, device="cpu"):
        all_patches = []
        for gen_idx in range(self.n_generators):
            weights = self.q_params[gen_idx]
            # [::anc_stride] = 보조큐빗이 0인 경우의 확률만 선택 → patch 밝기 총합(0~1)도 학습 대상이 됨
            batch_patches = [self.circuit(torch.rand(self.n_qubits, device=device) * math.pi, weights)[::self.anc_stride]
                              for _ in range(batch_size)]
            all_patches.append(torch.stack(batch_patches))
        return torch.cat(all_patches, dim=1).float()

    @property
    def output_dim(self):
        return self.n_generators * self.patch_dim


class ClassicalDiscriminator(nn.Module):
    '''HAM10000 QGAN 논문 Table 4 그대로 — 5층 완전연결, ReLU, sigmoid 출력.'''

    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


def train_step(generator, discriminator, real_batch, opt_g, opt_d, device="cpu"):
    bce = nn.BCELoss()
    batch_size = real_batch.shape[0]
    real_labels = torch.ones(batch_size, 1, device=device)
    fake_labels = torch.zeros(batch_size, 1, device=device)

    opt_d.zero_grad()
    real_pred = discriminator(real_batch)
    loss_d_real = bce(real_pred, real_labels)
    fake_batch = generator(batch_size, device=device).detach()
    fake_pred = discriminator(fake_batch)
    loss_d_fake = bce(fake_pred, fake_labels)
    loss_d = loss_d_real + loss_d_fake
    loss_d.backward()
    opt_d.step()

    opt_g.zero_grad()
    fake_batch = generator(batch_size, device=device)
    fake_pred = discriminator(fake_batch)
    loss_g = bce(fake_pred, real_labels)
    loss_g.backward()
    opt_g.step()
    return loss_g.item(), loss_d.item()

print("QGAN 모델 정의 완료")

## 5. QGAN 학습 실행

`--epochs` 등은 `train_qgan_checkpoint.py`의 CLI 인자와 동일한 의미입니다. 시간을 보면서
`EPOCHS`를 조절하세요(원 논문 2000epoch=1~1.5시간, 강의용은 훨씬 적은 epoch로도 패턴 확인 가능).

In [ ]:
TARGET_CLASS = "df"     # 논문과 동일 — 소수클래스
IMG_SIZE = 8            # 8x8=64px (2일차/3일차 노트북과 반드시 동일해야 함)
N_GENERATORS = 4
Q_DEPTH = 4
EPOCHS = 300
BATCH_SIZE = 8
LR_G = 0.05
LR_D = 0.001           # 판별자가 너무 빨리 이기지 않도록 낮춤(0.01에서는 판별자 압승 → 생성 품질 저하)

# 서브제너레이터 N_GENERATORS개가 patch를 이어붙여 전체 이미지(IMG_SIZE^2 픽셀)를 만드는 구조이므로,
# 생성자 총 출력차원(N_GENERATORS * 2^n_data_qubits)이 실제 이미지 픽셀수와 같아야 한다.
# (n_generators를 고려 안 하고 n_data_qubits=log2(IMG_SIZE^2)로만 잡으면 판별자에 넣는 real/fake
#  이미지 차원이 달라져 학습 즉시 shape 에러가 남 — 실행검증 중 실제로 발견해 이 계산식으로 수정함.)
total_pixels = IMG_SIZE ** 2
assert total_pixels % N_GENERATORS == 0, f"IMG_SIZE^2({total_pixels})가 N_GENERATORS({N_GENERATORS})로 나누어떨어져야 함"
patch_pixels = total_pixels // N_GENERATORS
n_data_qubits = int(np.log2(patch_pixels))
assert 2 ** n_data_qubits == patch_pixels, f"patch당 픽셀수({patch_pixels})가 2의 거듭제곱이어야 함"
print(f"[설정] target={TARGET_CLASS}, img_size={IMG_SIZE}x{IMG_SIZE}({total_pixels}px), "
      f"n_generators={N_GENERATORS}, patch당 {patch_pixels}px → n_data_qubits={n_data_qubits}, q_depth={Q_DEPTH}")

print("데이터 로드 + 세그멘테이션 전처리...")
target_imgs = build_dataset(metadata, IMG_DIR, TARGET_CLASS, IMG_SIZE)
print(f"  {TARGET_CLASS} 클래스 이미지: {len(target_imgs)}장")
assert len(target_imgs) > 0, "데이터가 없습니다 — Kaggle 다운로드가 정상적으로 됐는지 확인하세요."

real_flat = torch.tensor(target_imgs.reshape(len(target_imgs), -1))
# real 이미지를 확률 척도로 맞춤(이미지 1장 픽셀 합=1). 생성자 출력도 patch별 '보조큐빗=0' 확률의 합이라
# 전체 합이 0~N_GENERATORS 범위에서 학습되며, 판별자를 속이려면 real과 같은 총합·patch별 분포를 배워야 함.
real_flat = real_flat / real_flat.sum(dim=1, keepdim=True).clamp(min=1e-6)

gen = PatchQuantumGenerator(n_generators=N_GENERATORS, n_data_qubits=n_data_qubits, n_ancillas=1, q_depth=Q_DEPTH)
disc = ClassicalDiscriminator(input_dim=gen.output_dim)
opt_g = torch.optim.Adam(gen.parameters(), lr=LR_G)
opt_d = torch.optim.Adam(disc.parameters(), lr=LR_D)

t0 = time.time()
for epoch in range(EPOCHS):
    idx = torch.randint(0, len(real_flat), (min(BATCH_SIZE, len(real_flat)),))
    batch = real_flat[idx]
    lg, ld = train_step(gen, disc, batch, opt_g, opt_d)
    if (epoch + 1) % max(1, EPOCHS // 20) == 0:
        elapsed = time.time() - t0
        print(f"  epoch {epoch+1}/{EPOCHS}  loss_g={lg:.4f}  loss_d={ld:.4f}  경과={elapsed/60:.1f}분")

print(f"QGAN 학습 완료 — 총 소요 {(time.time()-t0)/60:.1f}분")

## 6. QGAN 체크포인트 저장 (Google Drive)

3일차 노트북이 그대로 불러올 수 있도록 `day3/data_pipeline.py`가 저장하는
Drive 경로(`/content/drive/MyDrive/동아대_QML강의/checkpoints/`)와 동일한 위치에 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/동아대_QML강의/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

qgan_ckpt_path = f"{SAVE_DIR}/qgan_{TARGET_CLASS}_{IMG_SIZE}px.pt"
torch.save({
    "generator_state_dict": gen.state_dict(),
    "config": {
        "n_generators": N_GENERATORS, "n_data_qubits": n_data_qubits,
        "n_ancillas": 1, "q_depth": Q_DEPTH, "img_size": IMG_SIZE,
        "target_class": TARGET_CLASS,
    },
}, qgan_ckpt_path)
print(f"저장됨: {qgan_ckpt_path}")

## 7. CNN 분류기 정의 (`cnn_classifier.py`와 동일 로직)

3일차 노트북의 `PaperCNNClassifier`와 완전히 동일한 클래스입니다 — 여기서 학습한 `state_dict`가
그대로 로드되어야 하므로 구조를 절대 바꾸지 마세요.

In [ ]:
class PaperCNNClassifier(nn.Module):
    '''논문 Table 2(conv 4층+BN+maxpool) / Table 1(FC+dropout2+sigmoid)을 입력 크기에
    맞춰 적응적으로 구성 — 8x8처럼 아주 작은 입력에서도 죽지 않도록 풀링 횟수를 자동 제한.'''

    def __init__(self, in_channels=1, img_size=64):
        super().__init__()
        chs_full = [in_channels, 16, 32, 64, 128]
        n_layers, size = 0, img_size
        while size >= 2 and n_layers < 4:
            size //= 2; n_layers += 1
        n_layers = max(n_layers, 1)
        chs = chs_full[: n_layers + 1]
        layers, size = [], img_size
        for i in range(n_layers):
            do_pool = size >= 2
            layers += [nn.Conv2d(chs[i], chs[i+1], 3, 1, 1), nn.BatchNorm2d(chs[i+1]), nn.ReLU()]
            if do_pool:
                layers.append(nn.MaxPool2d(2, 2)); size //= 2
        self.conv = nn.Sequential(*layers)
        flat_dim = chs[-1] * size * size
        self.fc = nn.Sequential(nn.Linear(flat_dim, 128), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))


def train_classifier(model, train_loader, val_loader, epochs=20, lr=1.5e-3, device="cpu"):
    '''논문 Table 1: ADAM, lr=1.5e-3, BCE loss.'''
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCELoss()
    history = {"train_loss": [], "val_loss": [], "val_auc": []}
    model.to(device)
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = bce(pred, yb)
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses, all_pred, all_true = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_losses.append(bce(pred, yb).item())
                all_pred.extend(pred.cpu().numpy().flatten().tolist())
                all_true.extend(yb.cpu().numpy().flatten().tolist())

        val_auc = roc_auc_score(all_true, all_pred) if len(set(all_true)) > 1 else float("nan")
        history["train_loss"].append(sum(train_losses) / len(train_losses))
        history["val_loss"].append(sum(val_losses) / len(val_losses))
        history["val_auc"].append(val_auc)
    return history


def evaluate(model, loader, device="cpu"):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy().flatten()
            all_pred.extend(pred.tolist())
            all_true.extend(yb.numpy().flatten().tolist())
    binary_pred = [1 if p > 0.5 else 0 for p in all_pred]
    return {
        "accuracy": accuracy_score(all_true, binary_pred),
        "auc_roc": roc_auc_score(all_true, all_pred) if len(set(all_true)) > 1 else float("nan"),
    }

print("CNN 분류기 정의 완료")

## 8. 증강 전/후 데이터 준비 — held-out 검증셋 분할

**정직한 비교 원칙**: 검증셋은 증강 전/후 모델에 **동일하게** 쓰고, 합성 이미지는 학습셋에만 넣습니다
(검증셋에 합성 이미지가 섞이면 "증강이 실제로 일반화 성능을 높였는지"를 확인할 수 없습니다).

In [ ]:
MAJORITY_CLASS = "nv"

def split_train_val(imgs, val_frac=0.2, seed=0):
    n = len(imgs)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    n_val = max(1, int(round(n * val_frac)))
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return imgs[train_idx], imgs[val_idx]

def make_loader(pos_imgs, neg_imgs, batch_size=8):
    x = np.concatenate([pos_imgs, neg_imgs], axis=0)[:, None, :, :]
    y = np.concatenate([np.ones(len(pos_imgs)), np.zeros(len(neg_imgs))])
    ds = TensorDataset(torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1))
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

minority_imgs = target_imgs  # 5번 셀에서 이미 만든 DF 전처리 데이터 재사용
majority_imgs = build_dataset(metadata, IMG_DIR, MAJORITY_CLASS, IMG_SIZE, max_n=len(minority_imgs) * 3)
print(f"소수({TARGET_CLASS}): {len(minority_imgs)}장, 다수({MAJORITY_CLASS}): {len(majority_imgs)}장")

minority_train, minority_val = split_train_val(minority_imgs)
majority_train, majority_val = split_train_val(majority_imgs)
print(f"train/val 분할: 소수 train={len(minority_train)}/val={len(minority_val)}, "
      f"다수 train={len(majority_train)}/val={len(majority_val)}")

n_synth = max(0, len(majority_train) - len(minority_train))
# 생성자 출력(확률 척도)을 실제 전처리 이미지의 픽셀 척도(0~1 밝기)로 환산 — 환산하지 않으면
# 합성 이미지가 실제 이미지와 밝기 수준이 달라, CNN이 '밝기'만으로 합성 이미지를 구분하게 됨
with torch.no_grad():
    synth = gen(n_synth).numpy() if n_synth > 0 else np.zeros((0, IMG_SIZE * IMG_SIZE))
pixel_scale = minority_train.reshape(len(minority_train), -1).sum(axis=1).mean() / max(synth.sum(axis=1).mean(), 1e-9) if n_synth > 0 else 1.0
synth = (synth * pixel_scale).reshape(-1, IMG_SIZE, IMG_SIZE)
synth = synth.astype(np.float32)
print(f"합성 이미지 픽셀 척도 환산: x{pixel_scale:.3f} (실제 평균 밝기 {minority_train.mean():.3f} / 합성 평균 밝기 {synth.mean() if n_synth else 0:.3f})")
minority_train_augmented = np.concatenate([minority_train, synth], axis=0)
print(f"생성된 합성 이미지: {n_synth}장 (train에만 추가, val에는 추가 안 함)")

## 9. CNN 학습·평가 (증강 전 vs 증강 후, 같은 검증셋)

In [ ]:
CNN_EPOCHS = 20

loader_before_train = make_loader(minority_train, majority_train)
loader_after_train = make_loader(minority_train_augmented, majority_train)
loader_val = make_loader(minority_val, majority_val)

model_before = PaperCNNClassifier(in_channels=1, img_size=IMG_SIZE)
hist_before = train_classifier(model_before, loader_before_train, loader_val, epochs=CNN_EPOCHS)
metrics_before = evaluate(model_before, loader_val)
print(f"[증강 전] val accuracy={metrics_before['accuracy']:.3f}  val AUC-ROC={metrics_before['auc_roc']:.3f}")

model_after = PaperCNNClassifier(in_channels=1, img_size=IMG_SIZE)
hist_after = train_classifier(model_after, loader_after_train, loader_val, epochs=CNN_EPOCHS)
metrics_after = evaluate(model_after, loader_val)
print(f"[증강 후] val accuracy={metrics_after['accuracy']:.3f}  val AUC-ROC={metrics_after['auc_roc']:.3f}")

## 10. 결과 저장 (Google Drive)

3일차 노트북 셀 2("사전학습 체크포인트 로드")가 그대로 불러오는 파일 3종을 저장합니다.

In [ ]:
torch.save({"model_state_dict": model_before.state_dict(), "img_size": IMG_SIZE},
           f"{SAVE_DIR}/cnn_before.pt")
torch.save({"model_state_dict": model_after.state_dict(), "img_size": IMG_SIZE},
           f"{SAVE_DIR}/cnn_after.pt")

with open(f"{SAVE_DIR}/comparison_result.json", "w", encoding="utf-8") as f:
    json.dump({
        "minority_class": TARGET_CLASS, "majority_class": MAJORITY_CLASS,
        "n_minority_real": len(minority_imgs), "n_synthetic": n_synth,
        "n_minority_train": len(minority_train), "n_minority_val": len(minority_val),
        "n_majority_train": len(majority_train), "n_majority_val": len(majority_val),
        "eval_set": "held_out_validation",
        "before": metrics_before, "after": metrics_after,
        "history_before": hist_before, "history_after": hist_after,
    }, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {SAVE_DIR}/")
print("  - qgan_{}_{}px.pt".format(TARGET_CLASS, IMG_SIZE))
print("  - cnn_before.pt, cnn_after.pt, comparison_result.json")

## 11. 실습 자산 묶기 → 저장소 `day3/assets/`에 커밋

3일차 실습 노트북은 강의 GitHub 저장소(https://github.com/hyunhp/skin_disease_qml)를 `git clone`해서
`day3/assets/`의 파일을 불러옵니다. 아래 셀이 배포 자산을 **flat 구조**(하위 폴더 없이)로 모아 zip으로 내려받게 해줍니다 —
3일차 노트북이 `f"{ASSET_DIR}/qgan_df_8px.pt"`처럼 최상위 경로로 접근하기 때문에 구조를 바꾸면 안 됩니다.

포함 파일: `qgan_df_8px.pt`, `cnn_before.pt`, `cnn_after.pt`, `comparison_result.json`,
`df_preprocessed.npy`, `nv_preprocessed.npy`(2일차 전처리 결과 — 생성 이미지와 비교용).

**내려받은 뒤(로컬 PC에서)**: zip을 풀어 저장소의 `day3/assets/`에 6개 파일을 넣고 커밋·푸시합니다.
```bash
git add day3/assets
git commit -m "Add day3 pretrained assets"
git push
```
Colab에서 직접 push하려면 GitHub 토큰을 노트북에 넣어야 하므로 권장하지 않습니다.

In [ ]:
# 배포 자산을 flat 구조로 스테이징 (3일차 노트북이 최상위 경로로 접근함)
import shutil
STAGE_DIR = "/content/qml_assets"
shutil.rmtree(STAGE_DIR, ignore_errors=True)
os.makedirs(STAGE_DIR, exist_ok=True)

for fname in [f"qgan_{TARGET_CLASS}_{IMG_SIZE}px.pt", "cnn_before.pt", "cnn_after.pt", "comparison_result.json"]:
    shutil.copy(f"{SAVE_DIR}/{fname}", f"{STAGE_DIR}/{fname}")

# 2일차 전처리 결과 — 3일차 노트북이 '실제 DF 원본' 비교용으로 로드함
np.save(f"{STAGE_DIR}/df_preprocessed.npy", minority_imgs)
np.save(f"{STAGE_DIR}/nv_preprocessed.npy", majority_imgs)

total_mb = sum(os.path.getsize(f"{STAGE_DIR}/{f}") for f in os.listdir(STAGE_DIR)) / 1024 / 1024
print("스테이징 완료:", sorted(os.listdir(STAGE_DIR)), f"(총 {total_mb:.1f}MB)")
assert total_mb < 50, "자산이 50MB를 넘음 — GitHub 권장 파일 크기를 넘기 전에 설정(IMG_SIZE 등)을 확인할 것"

In [ ]:
# zip으로 묶어 내려받기 (브라우저 다운로드 창이 뜸 — Drive의 checkpoints 폴더에도 원본이 남아 있음)
zip_path = shutil.make_archive("/content/day3_assets", "zip", STAGE_DIR)
print("zip 생성:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Colab 환경이 아니라 자동 다운로드는 생략함 — 위 경로의 zip을 직접 복사할 것")
print()
print("다음 작업(로컬 PC): zip을 풀어 저장소 day3/assets/에 6개 파일을 넣고 git add → commit → push")

---
## 체크리스트
- [ ] QGAN 학습 완료 (loss 추이 확인 — 발산하지 않았는지)
- [ ] CNN 증강 전/후 둘 다 **같은 held-out 검증셋**으로 평가됐는지 확인 (`eval_set: held_out_validation`)
- [ ] 체크포인트 3종 + json이 Drive `동아대_QML강의/checkpoints/`에 저장됐는지 확인
- [ ] `day3_assets.zip`을 내려받아 저장소 `day3/assets/`에 넣고 커밋·푸시 완료
- [ ] 3일차 실습 노트북을 Colab에서 **처음부터 끝까지** 실행해 git clone 경로로 정상 로드되는지 리허설

**다음 단계**: `day3/day3_practice.ipynb`를 Colab에서 열어 1번 셀부터
끝까지 실행해서, 푸시한 자산으로 정상 동작하는지 리허설할 것.
